# 03C Feature Importance and Model Interpretation

> 🟢 **Level A · Required**

Learn `correlation → impurity importance → permutation importance → SHAP`. Feature importance is not causality.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline


## Four different questions
Pearson correlation measures marginal linear association; impurity importance measures training-time split gains; permutation measures held-out score loss after shuffling a feature; SHAP attributes predictions relative to a model baseline. Correlated descriptors can share or mask importance. Negative permutation importance can reflect noise or overfitting. None establishes causality; do not use the final test repeatedly to redesign features.


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); y=df[target]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
imputer=SimpleImputer(strategy='median')
Xtr=pd.DataFrame(imputer.fit_transform(Xtr),columns=features,index=Xtr.index)
Xte=pd.DataFrame(imputer.transform(Xte),columns=features,index=Xte.index)
rf=RandomForestRegressor(n_estimators=400,random_state=42,n_jobs=-1).fit(Xtr,ytr)
display(df[features+[target]].corr(numeric_only=True)[target].drop(target).sort_values(key=abs,ascending=False))
pd.Series(rf.feature_importances_,index=features).sort_values().plot.barh(); plt.show()
perm=permutation_importance(rf,Xte,yte,n_repeats=20,random_state=42,scoring='r2')
pd.Series(perm.importances_mean,index=features).sort_values().plot.barh(); plt.show()


In [ ]:
!pip -q install shap
import shap
sample=Xte.iloc[:200]
shap.summary_plot(shap.TreeExplainer(rf).shap_values(sample),sample)


Interpret rankings in the context of adsorption physics and correlated descriptors.

### Completion criterion
Compare multiple importance approaches and distinguish predictive association from causality.


## Report stability, not only a ranking


In [ ]:
importance_table = pd.DataFrame({'feature': features, 'mean_score_drop': perm.importances_mean, 'repeat_std': perm.importances_std})
display(importance_table.sort_values('mean_score_drop', ascending=False))


## Exercise
Compare the top three features across methods. Explain one disagreement using correlated pore descriptors. Formulate a chemical hypothesis and propose an independent simulation or experiment to test it.


## Sources and further reading
[Dataset contracts / 数据使用约定](../../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
